# Advanced Python Set Operations
## A Tutorial-Style Problem Workbook with Complete Solutions

This notebook takes the same topic—Python set operations—but uses a slower, tutorial-oriented style.

Instead of jumping directly to compact final answers, each problem is broken into logical steps:

1. understand the membership rule;
2. inspect the data;
3. compute small intermediate sets;
4. combine those sets;
5. verify the result;
6. package the logic into a reusable solution.

The notebook contains many explanatory markdown cells because the goal is not only to obtain correct answers, but also to make the reasoning visible.

### What we will practice

We will use:

- intersections;
- unions;
- differences;
- symmetric differences;
- subset and superset tests;
- disjointness;
- set comprehensions;
- `frozenset`;
- dictionary key views;
- in-place set updates;
- frequency counting across many sets;
- brute-force search over subsets;
- graph and search applications;
- careful testing of unordered results.

### A note about output order

Sets are unordered collections.

Therefore, code such as:

```python
{3, 1, 2}
```

may be displayed in an order that should not be treated as part of the result.

Throughout this notebook, we will use `sorted(...)` when we want stable, easy-to-read output.

In [1]:
from collections import Counter, defaultdict
from collections.abc import Hashable, Iterable, Mapping, Sequence
from itertools import combinations
from typing import TypeVar

T = TypeVar("T", bound=Hashable)

def ordered(values: Iterable[T]) -> list[T]:
    """Return a deterministic representation for display and tests."""
    return sorted(values)

# Part 1 — A deeper warm-up

Before the larger problems, let us revisit the four central operations using a membership interpretation.

Suppose we have two sets:

- `A`: customers who purchased a laptop;
- `B`: customers who purchased a monitor.

In [2]:
A = {"Ana", "Bo", "Cy", "Di"}
B = {"Bo", "Di", "Eli", "Fay"}

ordered(A), ordered(B)

(['Ana', 'Bo', 'Cy', 'Di'], ['Bo', 'Di', 'Eli', 'Fay'])

The intersection answers:

> Who belongs to both groups?

In [3]:
ordered(A & B)

['Bo', 'Di']

The union answers:

> Who belongs to at least one group?

In [4]:
ordered(A | B)

['Ana', 'Bo', 'Cy', 'Di', 'Eli', 'Fay']

The difference `A - B` answers:

> Who belongs to `A`, but not to `B`?

In [5]:
ordered(A - B)

['Ana', 'Cy']

The symmetric difference answers:

> Who belongs to exactly one of the two groups?

In [6]:
ordered(A ^ B)

['Ana', 'Cy', 'Eli', 'Fay']

We can verify the symmetric-difference identity:

```python
A ^ B == (A | B) - (A & B)
```

In [7]:
assert A ^ B == (A | B) - (A & B)
ordered((A | B) - (A & B))

['Ana', 'Cy', 'Eli', 'Fay']

# Problem 1 — Advanced course-enrollment analysis

A training company runs three advanced courses:

- Python;
- SQL;
- Cloud.

We want to identify several useful groups:

1. students taking all three courses;
2. students taking at least one course;
3. students taking exactly one course;
4. students taking exactly two courses;
5. students taking Python but not Cloud.

In [8]:
python_students = {"Ana", "Bo", "Cy", "Di", "Eli", "Gus"}
sql_students = {"Bo", "Cy", "Eli", "Fay", "Gus"}
cloud_students = {"Cy", "Di", "Eli", "Gus", "Hana"}

## Step 1 — Find students taking all three courses

A student must be present in every set.

That means we need a three-way intersection.

In [9]:
all_three = python_students & sql_students & cloud_students
ordered(all_three)

['Cy', 'Eli', 'Gus']

## Step 2 — Find students taking at least one course

This is the union of all three sets.

In [10]:
at_least_one = python_students | sql_students | cloud_students
ordered(at_least_one)

['Ana', 'Bo', 'Cy', 'Di', 'Eli', 'Fay', 'Gus', 'Hana']

## Step 3 — Find students taking exactly one course

For Python-only students, remove everyone who appears in SQL or Cloud.

We repeat the same idea for the other courses.

In [11]:
python_only = python_students - sql_students - cloud_students
sql_only = sql_students - python_students - cloud_students
cloud_only = cloud_students - python_students - sql_students

ordered(python_only), ordered(sql_only), ordered(cloud_only)

(['Ana'], ['Fay'], ['Hana'])

The students taking exactly one course are the union of those three exclusive groups.

In [12]:
exactly_one = python_only | sql_only | cloud_only
ordered(exactly_one)

['Ana', 'Fay', 'Hana']

## Step 4 — Find students taking exactly two courses

We first find pairwise intersections.

Then we remove students who belong to all three courses.

In [13]:
python_sql_only = (python_students & sql_students) - cloud_students
python_cloud_only = (python_students & cloud_students) - sql_students
sql_cloud_only = (sql_students & cloud_students) - python_students

ordered(python_sql_only), ordered(python_cloud_only), ordered(sql_cloud_only)

(['Bo'], ['Di'], [])

In [14]:
exactly_two = python_sql_only | python_cloud_only | sql_cloud_only
ordered(exactly_two)

['Bo', 'Di']

## Step 5 — Find Python students who are not in Cloud

In [15]:
python_not_cloud = python_students - cloud_students
ordered(python_not_cloud)

['Ana', 'Bo']

## Complete reusable solution

In [16]:
def course_membership_summary(
    python_students: set[str],
    sql_students: set[str],
    cloud_students: set[str],
) -> dict[str, set[str]]:
    all_three = python_students & sql_students & cloud_students

    python_only = python_students - sql_students - cloud_students
    sql_only = sql_students - python_students - cloud_students
    cloud_only = cloud_students - python_students - sql_students

    exactly_one = python_only | sql_only | cloud_only

    exactly_two = (
        ((python_students & sql_students) - cloud_students)
        | ((python_students & cloud_students) - sql_students)
        | ((sql_students & cloud_students) - python_students)
    )

    return {
        "all_three": all_three,
        "at_least_one": python_students | sql_students | cloud_students,
        "exactly_one": exactly_one,
        "exactly_two": exactly_two,
        "python_not_cloud": python_students - cloud_students,
    }

course_summary = course_membership_summary(
    python_students,
    sql_students,
    cloud_students,
)

{k: ordered(v) for k, v in course_summary.items()}

{'all_three': ['Cy', 'Eli', 'Gus'],
 'at_least_one': ['Ana', 'Bo', 'Cy', 'Di', 'Eli', 'Fay', 'Gus', 'Hana'],
 'exactly_one': ['Ana', 'Fay', 'Hana'],
 'exactly_two': ['Bo', 'Di'],
 'python_not_cloud': ['Ana', 'Bo']}

In [17]:
assert course_summary["all_three"] == {"Cy", "Eli", "Gus"}
assert course_summary["exactly_one"] == {"Ana", "Fay", "Hana"}
assert course_summary["exactly_two"] == {"Bo", "Di"}

# Problem 2 — Exactly `k` groups

The previous problem had exactly three sets.

Now suppose we have an arbitrary number of project teams.

We want a general function that returns the people appearing in exactly `k` teams.

A direct set expression becomes difficult when the number of sets is not fixed, so we will combine sets with counting.

In [18]:
teams = [
    {"Ana", "Bo", "Cy", "Di"},
    {"Bo", "Cy", "Eli"},
    {"Cy", "Di", "Eli", "Fay"},
    {"Cy", "Fay", "Gus"},
]

## Step 1 — Count each membership once per team

Because each team is already a set, each person appears at most once inside a team.

We can flatten the team memberships and count names.

In [19]:
membership_counts = Counter(
    person
    for team in teams
    for person in team
)

membership_counts

Counter({'Cy': 4, 'Di': 2, 'Bo': 2, 'Eli': 2, 'Fay': 2, 'Ana': 1, 'Gus': 1})

## Step 2 — Select people whose count equals `k`

In [20]:
exactly_two_teams = {
    person
    for person, count in membership_counts.items()
    if count == 2
}

ordered(exactly_two_teams)

['Bo', 'Di', 'Eli', 'Fay']

## Step 3 — Generalize the logic

We also validate that `k` is positive.

In [21]:
def members_in_exactly_k_groups(
    groups: Iterable[Iterable[T]],
    k: int,
) -> set[T]:
    if k <= 0:
        raise ValueError("k must be positive")

    normalized_groups = [set(group) for group in groups]
    counts = Counter(
        item
        for group in normalized_groups
        for item in group
    )

    return {
        item
        for item, count in counts.items()
        if count == k
    }

ordered(members_in_exactly_k_groups(teams, 2))

['Bo', 'Di', 'Eli', 'Fay']

In [22]:
assert members_in_exactly_k_groups(teams, 1) == {"Ana", "Gus"}
assert members_in_exactly_k_groups(teams, 2) == {"Bo", "Di", "Eli", "Fay"}
assert members_in_exactly_k_groups(teams, 4) == {"Cy"}

# Problem 3 — Feature-flag rollout safety

A company is gradually enabling a new feature.

We have:

- an allowlist;
- a denylist;
- a beta-tester group;
- an employee group.

Rules:

1. denied users must never receive the feature;
2. allowlisted users receive the feature unless denied;
3. beta testers and employees also receive it unless denied;
4. we want to know which eligible users are not yet active.

In [23]:
allowlist = {"u1", "u2", "u3", "u7"}
denylist = {"u3", "u8"}
beta_testers = {"u2", "u4", "u5"}
employees = {"u5", "u6", "u8"}
currently_active = {"u1", "u2", "u5"}

## Step 1 — Combine all sources of eligibility

Eligibility begins as the union of:

- allowlist;
- beta testers;
- employees.

In [24]:
potentially_eligible = allowlist | beta_testers | employees
ordered(potentially_eligible)

['u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8']

## Step 2 — Apply the denylist

Explicit denial overrides every source of access.

In [25]:
eligible = potentially_eligible - denylist
ordered(eligible)

['u1', 'u2', 'u4', 'u5', 'u6', 'u7']

## Step 3 — Find eligible users not yet active

In [26]:
eligible_not_active = eligible - currently_active
ordered(eligible_not_active)

['u4', 'u6', 'u7']

## Step 4 — Detect incorrectly active users

These users are active but not eligible.

In [27]:
incorrectly_active = currently_active - eligible
ordered(incorrectly_active)

[]

## Complete solution

In [28]:
def feature_rollout_audit(
    allowlist: set[str],
    denylist: set[str],
    beta_testers: set[str],
    employees: set[str],
    currently_active: set[str],
) -> dict[str, set[str]]:
    eligible = (allowlist | beta_testers | employees) - denylist

    return {
        "eligible": eligible,
        "eligible_not_active": eligible - currently_active,
        "incorrectly_active": currently_active - eligible,
        "denied_but_active": denylist & currently_active,
    }

rollout = feature_rollout_audit(
    allowlist,
    denylist,
    beta_testers,
    employees,
    currently_active,
)

{k: ordered(v) for k, v in rollout.items()}

{'eligible': ['u1', 'u2', 'u4', 'u5', 'u6', 'u7'],
 'eligible_not_active': ['u4', 'u6', 'u7'],
 'incorrectly_active': [],
 'denied_but_active': []}

In [29]:
assert rollout["eligible"] == {"u1", "u2", "u4", "u5", "u6", "u7"}
assert rollout["eligible_not_active"] == {"u4", "u6", "u7"}
assert rollout["incorrectly_active"] == set()

# Problem 4 — Database schema migration audit

A table is moving from an old schema to a new schema.

We want to classify column names as:

- unchanged;
- added;
- removed;
- renamed candidates.

The rename candidates are supplied separately as pairs.

In [30]:
old_columns = {
    "user_id",
    "first_name",
    "last_name",
    "email",
    "created_at",
    "status",
}

new_columns = {
    "user_id",
    "given_name",
    "family_name",
    "email",
    "created_at",
    "status",
    "country",
}

rename_candidates = {
    ("first_name", "given_name"),
    ("last_name", "family_name"),
}

## Step 1 — Unchanged columns

These appear in both schemas.

In [31]:
unchanged = old_columns & new_columns
ordered(unchanged)

['created_at', 'email', 'status', 'user_id']

## Step 2 — Raw removed and added columns

In [32]:
raw_removed = old_columns - new_columns
raw_added = new_columns - old_columns

ordered(raw_removed), ordered(raw_added)

(['first_name', 'last_name'], ['country', 'family_name', 'given_name'])

## Step 3 — Validate rename candidates

A valid rename candidate should have:

- an old name in the removed set;
- a new name in the added set.

In [33]:
valid_renames = {
    pair
    for pair in rename_candidates
    if pair[0] in raw_removed and pair[1] in raw_added
}

sorted(valid_renames)

[('first_name', 'given_name'), ('last_name', 'family_name')]

## Step 4 — Remove renamed columns from true additions and removals

In [34]:
renamed_old = {old for old, new in valid_renames}
renamed_new = {new for old, new in valid_renames}

truly_removed = raw_removed - renamed_old
truly_added = raw_added - renamed_new

ordered(truly_removed), ordered(truly_added)

([], ['country'])

## Complete solution

In [35]:
def schema_migration_report(
    old_columns: set[str],
    new_columns: set[str],
    rename_candidates: set[tuple[str, str]],
) -> dict[str, object]:
    unchanged = old_columns & new_columns
    removed = old_columns - new_columns
    added = new_columns - old_columns

    valid_renames = {
        pair
        for pair in rename_candidates
        if pair[0] in removed and pair[1] in added
    }

    renamed_old = {old for old, _ in valid_renames}
    renamed_new = {new for _, new in valid_renames}

    return {
        "unchanged": unchanged,
        "renames": valid_renames,
        "added": added - renamed_new,
        "removed": removed - renamed_old,
    }

schema_report = schema_migration_report(
    old_columns,
    new_columns,
    rename_candidates,
)

schema_report

{'unchanged': {'created_at', 'email', 'status', 'user_id'},
 'renames': {('first_name', 'given_name'), ('last_name', 'family_name')},
 'added': {'country'},
 'removed': set()}

In [36]:
assert schema_report["unchanged"] == {
    "user_id", "email", "created_at", "status"
}
assert schema_report["added"] == {"country"}
assert schema_report["removed"] == set()

# Problem 5 — Pairwise disjoint partition validation

A partition of a universe must satisfy two conditions:

1. every pair of groups is disjoint;
2. the union of the groups equals the universe.

We will validate both conditions and report exact overlaps.

In [37]:
universe = set(range(1, 13))

groups = {
    "A": {1, 2, 3},
    "B": {4, 5, 6},
    "C": {7, 8, 9},
    "D": {10, 11, 12},
}

## Step 1 — Compare every pair

The `combinations` function gives every unordered pair once.

In [38]:
pair_overlaps = {}

for left, right in combinations(groups, 2):
    overlap = groups[left] & groups[right]
    if overlap:
        pair_overlaps[(left, right)] = overlap

pair_overlaps

{}

No overlaps means the groups are pairwise disjoint.

In [39]:
pairwise_disjoint = not pair_overlaps
pairwise_disjoint

True

## Step 2 — Compute total coverage

In [40]:
covered = set().union(*groups.values())
ordered(covered)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

## Step 3 — Find missing and extra values

In [41]:
missing = universe - covered
extra = covered - universe

ordered(missing), ordered(extra)

([], [])

## Complete validator

In [42]:
def validate_partition(
    universe: set[T],
    groups: Mapping[str, set[T]],
) -> dict[str, object]:
    overlaps: dict[tuple[str, str], set[T]] = {}

    for left, right in combinations(groups, 2):
        overlap = groups[left] & groups[right]
        if overlap:
            overlaps[(left, right)] = overlap

    covered = set().union(*groups.values()) if groups else set()

    return {
        "valid": not overlaps and covered == universe,
        "overlaps": overlaps,
        "missing": universe - covered,
        "extra": covered - universe,
    }

partition_report = validate_partition(universe, groups)
partition_report

{'valid': True, 'overlaps': {}, 'missing': set(), 'extra': set()}

In [43]:
assert partition_report["valid"] is True

bad_groups = dict(groups)
bad_groups["D"] = {9, 10, 11}

bad_report = validate_partition(universe, bad_groups)

assert bad_report["valid"] is False
assert bad_report["overlaps"][("C", "D")] == {9}
assert bad_report["missing"] == {12}

# Problem 6 — Duplicate-event detection across logs

Three services emit event IDs.

We need to identify:

- event IDs seen by every service;
- event IDs seen by only one service;
- IDs duplicated across at least two services;
- IDs present in an odd number of services.

In [44]:
service_a = {"e1", "e2", "e3", "e7", "e9"}
service_b = {"e2", "e3", "e4", "e7"}
service_c = {"e3", "e5", "e7", "e9"}

## Step 1 — Seen everywhere

In [45]:
seen_everywhere = service_a & service_b & service_c
ordered(seen_everywhere)

['e3', 'e7']

## Step 2 — Seen in only one service

We can reuse the exclusive-difference pattern.

In [46]:
only_a = service_a - service_b - service_c
only_b = service_b - service_a - service_c
only_c = service_c - service_a - service_b

seen_once = only_a | only_b | only_c
ordered(seen_once)

['e1', 'e4', 'e5']

## Step 3 — Seen in at least two services

One way is to union the three pairwise intersections.

In [47]:
seen_at_least_twice = (
    (service_a & service_b)
    | (service_a & service_c)
    | (service_b & service_c)
)

ordered(seen_at_least_twice)

['e2', 'e3', 'e7', 'e9']

## Step 4 — Odd parity

For three sets, chained XOR keeps values seen once or three times.

In [48]:
odd_parity = service_a ^ service_b ^ service_c
ordered(odd_parity)

['e1', 'e3', 'e4', 'e5', 'e7']

This is an important distinction:

- exactly once is not the same as odd parity;
- an event in all three services has count three, which is odd.

## Complete solution using counts

In [49]:
def log_membership_report(
    logs: Sequence[set[str]],
) -> dict[str, set[str]]:
    if not logs:
        return {
            "everywhere": set(),
            "exactly_once": set(),
            "at_least_twice": set(),
            "odd_parity": set(),
        }

    counts = Counter(
        event
        for log in logs
        for event in log
    )

    return {
        "everywhere": set.intersection(*logs),
        "exactly_once": {
            event for event, count in counts.items() if count == 1
        },
        "at_least_twice": {
            event for event, count in counts.items() if count >= 2
        },
        "odd_parity": {
            event for event, count in counts.items() if count % 2 == 1
        },
    }

log_report = log_membership_report(
    [service_a, service_b, service_c]
)

{k: ordered(v) for k, v in log_report.items()}

{'everywhere': ['e3', 'e7'],
 'exactly_once': ['e1', 'e4', 'e5'],
 'at_least_twice': ['e2', 'e3', 'e7', 'e9'],
 'odd_parity': ['e1', 'e3', 'e4', 'e5', 'e7']}

In [50]:
assert log_report["everywhere"] == {"e3", "e7"}
assert log_report["exactly_once"] == {"e1", "e4", "e5"}
assert log_report["odd_parity"] == {"e1", "e3", "e4", "e5", "e7"}

# Problem 7 — Tag similarity and recommendations

We have articles represented by sets of tags.

We want to recommend articles similar to a target article.

We will use Jaccard similarity:

```text
size of intersection / size of union
```

In [51]:
article_tags = {
    "A": {"python", "sets", "hashing", "tutorial"},
    "B": {"python", "sets", "performance", "tutorial"},
    "C": {"sql", "joins", "database"},
    "D": {"python", "frozenset", "hashing", "advanced"},
    "E": {"sets", "math", "union", "intersection"},
}

## Step 1 — Compare two articles manually

In [52]:
target = article_tags["A"]
candidate = article_tags["B"]

shared_tags = target & candidate
combined_tags = target | candidate

ordered(shared_tags), ordered(combined_tags)

(['python', 'sets', 'tutorial'],
 ['hashing', 'performance', 'python', 'sets', 'tutorial'])

In [53]:
score_ab = len(shared_tags) / len(combined_tags)
score_ab

0.6

## Step 2 — Write a reusable similarity function

We define two empty sets as perfectly similar.

In [54]:
def jaccard_similarity(
    left: Iterable[T],
    right: Iterable[T],
) -> float:
    a = set(left)
    b = set(right)
    union = a | b

    if not union:
        return 1.0

    return len(a & b) / len(union)

## Step 3 — Score every candidate

In [55]:
scores = {}

for article_id, tags in article_tags.items():
    if article_id == "A":
        continue

    scores[article_id] = jaccard_similarity(
        article_tags["A"],
        tags,
    )

scores

{'B': 0.6, 'C': 0.0, 'D': 0.3333333333333333, 'E': 0.14285714285714285}

## Step 4 — Rank by score

In [56]:
ranked_articles = sorted(
    scores.items(),
    key=lambda item: (-item[1], item[0]),
)

ranked_articles

[('B', 0.6), ('D', 0.3333333333333333), ('E', 0.14285714285714285), ('C', 0.0)]

## Complete recommendation function

In [57]:
def recommend_by_tags(
    item_tags: Mapping[str, set[str]],
    target_id: str,
    *,
    minimum_score: float = 0.0,
) -> list[tuple[str, float]]:
    if target_id not in item_tags:
        raise KeyError(f"Unknown item: {target_id}")

    target_tags = item_tags[target_id]
    results = []

    for item_id, tags in item_tags.items():
        if item_id == target_id:
            continue

        score = jaccard_similarity(target_tags, tags)

        if score >= minimum_score:
            results.append((item_id, round(score, 3)))

    return sorted(
        results,
        key=lambda item: (-item[1], item[0]),
    )

recommendations = recommend_by_tags(
    article_tags,
    "A",
    minimum_score=0.15,
)

recommendations

[('B', 0.6), ('D', 0.333)]

In [58]:
assert recommendations[0][0] == "B"
assert recommendations[-1][0] == "D"

# Problem 8 — Boolean document search

We will build a small inverted index.

An inverted index maps each word to the set of document IDs containing that word.

In [59]:
documents = {
    1: "python sets provide fast membership tests",
    2: "python dictionaries map keys to values",
    3: "sets support union intersection and difference",
    4: "database indexes support fast search",
    5: "python search can use an inverted index",
}

## Step 1 — Build the index

We convert the words of each document to a set so that repeated words in one document are indexed only once.

In [60]:
inverted_index: dict[str, set[int]] = defaultdict(set)

for document_id, text in documents.items():
    unique_words = set(text.lower().split())

    for word in unique_words:
        inverted_index[word].add(document_id)

dict(inverted_index)

{'membership': {1},
 'provide': {1},
 'tests': {1},
 'fast': {1, 4},
 'python': {1, 2, 5},
 'sets': {1, 3},
 'values': {2},
 'keys': {2},
 'to': {2},
 'dictionaries': {2},
 'map': {2},
 'union': {3},
 'and': {3},
 'support': {3, 4},
 'difference': {3},
 'intersection': {3},
 'search': {4, 5},
 'indexes': {4},
 'database': {4},
 'inverted': {5},
 'use': {5},
 'can': {5},
 'an': {5},
 'index': {5}}

## Step 2 — Required terms

For documents containing both `python` and `sets`, use intersection.

In [61]:
python_docs = inverted_index.get("python", set())
sets_docs = inverted_index.get("sets", set())

required_result = python_docs & sets_docs
ordered(required_result)

[1]

## Step 3 — Alternative terms

For documents containing `union` or `difference`, use union.

In [62]:
alternative_result = (
    inverted_index.get("union", set())
    | inverted_index.get("difference", set())
)

ordered(alternative_result)

[3]

## Step 4 — Excluded terms

For Python documents that do not contain `dictionaries`, use difference.

In [63]:
excluded_result = (
    inverted_index.get("python", set())
    - inverted_index.get("dictionaries", set())
)

ordered(excluded_result)

[1, 5]

## Complete search helper

In [64]:
def search_documents(
    index: Mapping[str, set[int]],
    *,
    all_terms: Iterable[str] = (),
    any_terms: Iterable[str] = (),
    excluded_terms: Iterable[str] = (),
    universe: Iterable[int] = (),
) -> set[int]:
    all_terms = [term.lower() for term in all_terms]
    any_terms = [term.lower() for term in any_terms]
    excluded_terms = [term.lower() for term in excluded_terms]

    if all_terms:
        required = set.intersection(
            *(set(index.get(term, set())) for term in all_terms)
        )
    else:
        required = set(universe)

    if any_terms:
        alternatives = set.union(
            *(set(index.get(term, set())) for term in any_terms)
        )
        required &= alternatives

    excluded = set().union(
        *(index.get(term, set()) for term in excluded_terms)
    ) if excluded_terms else set()

    return required - excluded

search_result = search_documents(
    inverted_index,
    all_terms={"python"},
    any_terms={"sets", "search"},
    excluded_terms={"dictionaries"},
    universe=documents,
)

ordered(search_result)

[1, 5]

In [65]:
assert search_result == {1, 5}

# Problem 9 — Two-hop graph neighborhoods

A graph can be represented as a dictionary from a node to a set of neighboring nodes.

For a chosen node, we want:

- direct neighbors;
- nodes reachable in exactly two steps;
- nodes reachable in at most two steps;
- second-hop recommendation candidates excluding the node and its direct neighbors.

In [66]:
graph = {
    "A": {"B", "C"},
    "B": {"A", "D", "E"},
    "C": {"A", "D", "F"},
    "D": {"B", "C", "G"},
    "E": {"B"},
    "F": {"C", "G"},
    "G": {"D", "F"},
}

## Step 1 — Direct neighbors

In [67]:
node = "A"
direct = graph[node]
ordered(direct)

['B', 'C']

## Step 2 — Collect neighbors of direct neighbors

In [68]:
neighbors_of_neighbors = set()

for neighbor in direct:
    neighbors_of_neighbors.update(graph[neighbor])

ordered(neighbors_of_neighbors)

['A', 'D', 'E', 'F']

## Step 3 — Remove the starting node and direct neighbors

This leaves nodes at exactly distance two.

In [69]:
exactly_two_steps = neighbors_of_neighbors - direct - {node}
ordered(exactly_two_steps)

['D', 'E', 'F']

## Step 4 — Reusable solution

In [70]:
def two_hop_analysis(
    graph: Mapping[T, set[T]],
    node: T,
) -> dict[str, set[T]]:
    direct = set(graph[node])

    neighbors_of_neighbors = set().union(
        *(graph[neighbor] for neighbor in direct)
    ) if direct else set()

    exactly_two = neighbors_of_neighbors - direct - {node}

    return {
        "direct": direct,
        "exactly_two": exactly_two,
        "within_two": direct | exactly_two,
        "recommendations": exactly_two,
    }

two_hop = two_hop_analysis(graph, "A")

{k: ordered(v) for k, v in two_hop.items()}

{'direct': ['B', 'C'],
 'exactly_two': ['D', 'E', 'F'],
 'within_two': ['B', 'C', 'D', 'E', 'F'],
 'recommendations': ['D', 'E', 'F']}

In [71]:
assert two_hop["exactly_two"] == {"D", "E", "F"}

# Problem 10 — API capability negotiation

A client and server advertise supported capabilities.

We want to determine:

- capabilities supported by both;
- client requests unsupported by the server;
- server capabilities unused by the client;
- whether all required capabilities can be satisfied;
- a selected optional capability based on preference order.

In [72]:
client_supported = {
    "json",
    "compression",
    "streaming",
    "batch",
}

server_supported = {
    "json",
    "compression",
    "streaming",
    "xml",
    "retry",
}

required = {
    "json",
    "streaming",
}

optional_preference = [
    "batch",
    "compression",
    "retry",
]

## Step 1 — Common capabilities

In [73]:
common = client_supported & server_supported
ordered(common)

['compression', 'json', 'streaming']

## Step 2 — Unsupported client capabilities

In [74]:
unsupported_by_server = client_supported - server_supported
ordered(unsupported_by_server)

['batch']

## Step 3 — Check required capabilities

All required capabilities must be a subset of the common capabilities.

In [75]:
requirements_satisfied = required <= common
requirements_satisfied

True

## Step 4 — Select the first available optional capability

In [76]:
selected_optional = next(
    (
        capability
        for capability in optional_preference
        if capability in common
    ),
    None,
)

selected_optional

'compression'

## Complete solution

In [77]:
def negotiate_capabilities(
    client_supported: set[str],
    server_supported: set[str],
    required: set[str],
    optional_preference: Sequence[str],
) -> dict[str, object]:
    common = client_supported & server_supported

    selected_optional = next(
        (
            capability
            for capability in optional_preference
            if capability in common
        ),
        None,
    )

    return {
        "common": common,
        "unsupported_by_server": client_supported - server_supported,
        "unused_server_capabilities": server_supported - client_supported,
        "missing_required": required - common,
        "requirements_satisfied": required <= common,
        "selected_optional": selected_optional,
    }

negotiation = negotiate_capabilities(
    client_supported,
    server_supported,
    required,
    optional_preference,
)

negotiation

{'common': {'compression', 'json', 'streaming'},
 'unsupported_by_server': {'batch'},
 'unused_server_capabilities': {'retry', 'xml'},
 'missing_required': set(),
 'requirements_satisfied': True,
 'selected_optional': 'compression'}

In [78]:
assert negotiation["requirements_satisfied"] is True
assert negotiation["selected_optional"] == "compression"
assert negotiation["unsupported_by_server"] == {"batch"}

# Problem 11 — Dictionary key-view operations

Dictionary views are useful because key views behave like set-like collections.

We will compare configuration dictionaries.

In [79]:
default_config = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "timeout": 30,
}

user_config = {
    "host": "example.com",
    "debug": True,
    "theme": "dark",
}

## Step 1 — Shared keys

In [80]:
shared_keys = default_config.keys() & user_config.keys()
ordered(shared_keys)

['debug', 'host']

## Step 2 — Keys only in defaults

In [81]:
default_only_keys = default_config.keys() - user_config.keys()
ordered(default_only_keys)

['port', 'timeout']

## Step 3 — User-defined unknown keys

In [82]:
unknown_user_keys = user_config.keys() - default_config.keys()
ordered(unknown_user_keys)

['theme']

## Step 4 — Detect changed values among shared keys

In [83]:
changed_shared_keys = {
    key
    for key in shared_keys
    if default_config[key] != user_config[key]
}

ordered(changed_shared_keys)

['debug', 'host']

## Complete configuration audit

In [84]:
def configuration_audit(
    defaults: Mapping[str, object],
    user_values: Mapping[str, object],
) -> dict[str, set[str]]:
    shared = defaults.keys() & user_values.keys()

    return {
        "shared": set(shared),
        "default_only": defaults.keys() - user_values.keys(),
        "unknown_user_keys": user_values.keys() - defaults.keys(),
        "changed": {
            key
            for key in shared
            if defaults[key] != user_values[key]
        },
        "unchanged": {
            key
            for key in shared
            if defaults[key] == user_values[key]
        },
    }

config_report = configuration_audit(
    default_config,
    user_config,
)

{k: ordered(v) for k, v in config_report.items()}

{'shared': ['debug', 'host'],
 'default_only': ['port', 'timeout'],
 'unknown_user_keys': ['theme'],
 'changed': ['debug', 'host'],
 'unchanged': []}

In [85]:
assert config_report["changed"] == {"host", "debug"}
assert config_report["unknown_user_keys"] == {"theme"}

# Problem 12 — Immutable sets as dictionary keys

Suppose an undirected connection is identified by its two endpoints.

The connection `{A, B}` should be considered the same as `{B, A}`.

A normal set cannot be a dictionary key because it is mutable and unhashable.

A `frozenset` is a natural canonical representation.

In [86]:
edge_weights = {
    frozenset({"A", "B"}): 7,
    frozenset({"B", "C"}): 4,
    frozenset({"A", "C"}): 9,
}

## Step 1 — Order-independent lookup

In [87]:
edge_weights[frozenset({"B", "A"})]

7

## Step 2 — Validate that every edge has exactly two distinct endpoints

In [88]:
invalid_edges = {
    edge
    for edge in edge_weights
    if len(edge) != 2
}

invalid_edges

set()

## Step 3 — Find all nodes

In [89]:
all_nodes = set().union(*edge_weights)
ordered(all_nodes)

['A', 'B', 'C']

## Step 4 — Build an adjacency mapping

In [90]:
adjacency: dict[str, set[str]] = {
    node: set()
    for node in all_nodes
}

for edge in edge_weights:
    left, right = tuple(edge)
    adjacency[left].add(right)
    adjacency[right].add(left)

{k: ordered(v) for k, v in adjacency.items()}

{'A': ['B', 'C'], 'C': ['A', 'B'], 'B': ['A', 'C']}

## Complete helper

In [91]:
def build_weighted_undirected_graph(
    edge_weights: Mapping[frozenset[T], float],
) -> dict[T, dict[T, float]]:
    invalid = {
        edge
        for edge in edge_weights
        if len(edge) != 2
    }

    if invalid:
        raise ValueError(
            "Every edge must contain exactly two distinct endpoints"
        )

    nodes = set().union(*edge_weights) if edge_weights else set()

    graph: dict[T, dict[T, float]] = {
        node: {}
        for node in nodes
    }

    for edge, weight in edge_weights.items():
        left, right = tuple(edge)
        graph[left][right] = weight
        graph[right][left] = weight

    return graph

weighted_graph = build_weighted_undirected_graph(edge_weights)
weighted_graph

{'A': {'B': 7, 'C': 9}, 'C': {'B': 4, 'A': 9}, 'B': {'A': 7, 'C': 4}}

In [92]:
assert weighted_graph["A"]["B"] == 7
assert weighted_graph["B"]["A"] == 7

# Problem 13 — Small exact hitting-set problem

A hitting set is a set that intersects every constraint set.

For example, suppose each constraint lists acceptable servers for a job.

We want the smallest server selection that hits every constraint.

This is computationally expensive for large inputs, but brute force is suitable for a small educational example.

In [93]:
constraints = [
    {"S1", "S2"},
    {"S2", "S3"},
    {"S3", "S4"},
    {"S1", "S4"},
]

## Step 1 — Build the candidate universe

In [94]:
candidate_servers = set().union(*constraints)
ordered(candidate_servers)

['S1', 'S2', 'S3', 'S4']

## Step 2 — Define what it means to hit every constraint

A candidate selection is valid when it is not disjoint from any constraint.

In [95]:
def hits_every_constraint(
    selection: set[T],
    constraints: Sequence[set[T]],
) -> bool:
    return all(
        not selection.isdisjoint(constraint)
        for constraint in constraints
    )

## Step 3 — Try combinations from smallest to largest

The first valid size is minimal.

In [96]:
smallest_hitting_sets = []

for size in range(len(candidate_servers) + 1):
    for combo in combinations(sorted(candidate_servers), size):
        selection = set(combo)

        if hits_every_constraint(selection, constraints):
            smallest_hitting_sets.append(selection)

    if smallest_hitting_sets:
        break

[ordered(selection) for selection in smallest_hitting_sets]

[['S1', 'S3'], ['S2', 'S4']]

## Complete solution

In [97]:
def minimum_hitting_sets(
    constraints: Sequence[set[T]],
) -> list[set[T]]:
    if not constraints:
        return [set()]

    universe = set().union(*constraints)

    for size in range(len(universe) + 1):
        solutions = []

        for combo in combinations(sorted(universe), size):
            candidate = set(combo)

            if all(
                not candidate.isdisjoint(constraint)
                for constraint in constraints
            ):
                solutions.append(candidate)

        if solutions:
            return solutions

    return []

minimum_hits = minimum_hitting_sets(constraints)
[ordered(solution) for solution in minimum_hits]

[['S1', 'S3'], ['S2', 'S4']]

In [98]:
assert all(len(solution) == 2 for solution in minimum_hits)
assert {frozenset(solution) for solution in minimum_hits} == {
    frozenset({"S1", "S3"}),
    frozenset({"S2", "S4"}),
}

# Problem 14 — Sudoku candidate sets

For one Sudoku cell, a digit is allowed when it is absent from:

- the cell's row;
- the cell's column;
- the cell's 3×3 box.

Set difference expresses this rule directly.

In [99]:
DIGITS = set(range(1, 10))

row_values = {1, 2, 4, 7}
column_values = {2, 3, 5, 8}
box_values = {1, 3, 6}

## Step 1 — Combine all used digits

In [100]:
used_digits = row_values | column_values | box_values
ordered(used_digits)

[1, 2, 3, 4, 5, 6, 7, 8]

## Step 2 — Subtract from the full digit universe

In [101]:
candidates = DIGITS - used_digits
ordered(candidates)

[9]

## Step 3 — Generalize

In [102]:
def sudoku_candidates(
    row_values: Iterable[int],
    column_values: Iterable[int],
    box_values: Iterable[int],
) -> set[int]:
    used = (
        set(row_values)
        | set(column_values)
        | set(box_values)
    ) - {0}

    return set(range(1, 10)) - used

candidate_result = sudoku_candidates(
    row_values,
    column_values,
    box_values,
)

ordered(candidate_result)

[9]

In [103]:
assert candidate_result == {9}
assert sudoku_candidates([], [], []) == set(range(1, 10))

# Problem 15 — Dataset drift across snapshots

A system records the set of active category labels each day.

We want to analyze how the category vocabulary changes over time.

In [104]:
snapshots = {
    "day_1": {"free", "paid", "trial", "education"},
    "day_2": {"free", "paid", "trial", "enterprise"},
    "day_3": {"free", "paid", "enterprise", "nonprofit"},
}

## Step 1 — Compare consecutive snapshots

In [105]:
day_1 = snapshots["day_1"]
day_2 = snapshots["day_2"]

added_day_2 = day_2 - day_1
removed_day_2 = day_1 - day_2
stable_day_2 = day_1 & day_2

ordered(added_day_2), ordered(removed_day_2), ordered(stable_day_2)

(['enterprise'], ['education'], ['free', 'paid', 'trial'])

## Step 2 — Find categories present every day

In [106]:
persistent_categories = set.intersection(*snapshots.values())
ordered(persistent_categories)

['free', 'paid']

## Step 3 — Find categories seen on any day

In [107]:
all_categories = set.union(*snapshots.values())
ordered(all_categories)

['education', 'enterprise', 'free', 'nonprofit', 'paid', 'trial']

## Step 4 — Find categories that appeared on exactly one day

In [108]:
category_counts = Counter(
    category
    for snapshot in snapshots.values()
    for category in snapshot
)

one_day_only = {
    category
    for category, count in category_counts.items()
    if count == 1
}

ordered(one_day_only)

['education', 'nonprofit']

## Complete drift report

In [109]:
def snapshot_drift(
    snapshots: Mapping[str, set[T]],
) -> dict[str, object]:
    labels = list(snapshots)

    transitions = {}

    for previous, current in zip(labels, labels[1:]):
        previous_set = snapshots[previous]
        current_set = snapshots[current]

        transitions[(previous, current)] = {
            "added": current_set - previous_set,
            "removed": previous_set - current_set,
            "stable": previous_set & current_set,
        }

    if snapshots:
        persistent = set.intersection(*snapshots.values())
        any_seen = set.union(*snapshots.values())
    else:
        persistent = set()
        any_seen = set()

    counts = Counter(
        item
        for snapshot in snapshots.values()
        for item in snapshot
    )

    return {
        "transitions": transitions,
        "persistent": persistent,
        "any_seen": any_seen,
        "exactly_one_snapshot": {
            item for item, count in counts.items() if count == 1
        },
    }

drift_report = snapshot_drift(snapshots)
drift_report

{'transitions': {('day_1', 'day_2'): {'added': {'enterprise'},
   'removed': {'education'},
   'stable': {'free', 'paid', 'trial'}},
  ('day_2', 'day_3'): {'added': {'nonprofit'},
   'removed': {'trial'},
   'stable': {'enterprise', 'free', 'paid'}}},
 'persistent': {'free', 'paid'},
 'any_seen': {'education', 'enterprise', 'free', 'nonprofit', 'paid', 'trial'},
 'exactly_one_snapshot': {'education', 'nonprofit'}}

In [110]:
assert drift_report["persistent"] == {"free", "paid"}
assert drift_report["exactly_one_snapshot"] == {
    "education", "nonprofit"
}

# Problem 16 — Tournament scheduling with disjoint participants

Each proposed match contains a set of participants.

Matches scheduled in the same time slot must have disjoint participant sets.

We will use a deterministic greedy strategy:

1. consider matches with fewer participants first;
2. break ties by match name;
3. accept a match only if it is disjoint from all already used participants.

In [111]:
matches = {
    "M1": {"A", "B"},
    "M2": {"C", "D"},
    "M3": {"B", "E"},
    "M4": {"F", "G"},
    "M5": {"D", "H"},
    "M6": {"I", "J", "K"},
}

## Step 1 — Track occupied participants

We begin with an empty set.

In [112]:
occupied = set()
selected = []

## Step 2 — Process matches in deterministic order

In [113]:
ordered_match_names = sorted(
    matches,
    key=lambda name: (len(matches[name]), name),
)

ordered_match_names

['M1', 'M2', 'M3', 'M4', 'M5', 'M6']

## Step 3 — Accept only disjoint matches

In [114]:
for match_name in ordered_match_names:
    participants = matches[match_name]

    if participants.isdisjoint(occupied):
        selected.append(match_name)
        occupied.update(participants)

selected, ordered(occupied)

(['M1', 'M2', 'M4', 'M6'], ['A', 'B', 'C', 'D', 'F', 'G', 'I', 'J', 'K'])

## Complete greedy scheduler

In [115]:
def select_disjoint_matches(
    matches: Mapping[str, set[T]],
) -> list[str]:
    selected = []
    occupied: set[T] = set()

    for match_name in sorted(
        matches,
        key=lambda name: (len(matches[name]), name),
    ):
        participants = matches[match_name]

        if participants.isdisjoint(occupied):
            selected.append(match_name)
            occupied.update(participants)

    return selected

selected_matches = select_disjoint_matches(matches)
selected_matches

['M1', 'M2', 'M4', 'M6']

In [116]:
for left, right in combinations(selected_matches, 2):
    assert matches[left].isdisjoint(matches[right])

# Problem 17 — Set identities as executable mathematics

Set algebra has laws similar to Boolean algebra.

We will verify several identities over every subset of a small universe.

The small universe lets us test all possibilities exhaustively.

In [117]:
small_universe = {0, 1, 2}

## Step 1 — Generate the powerset

The powerset contains every subset of a set.

We use `frozenset` so each subset can itself belong to a set.

In [118]:
def powerset(values: Iterable[T]) -> set[frozenset[T]]:
    items = tuple(dict.fromkeys(values))

    return {
        frozenset(combo)
        for size in range(len(items) + 1)
        for combo in combinations(items, size)
    }

all_subsets = [set(item) for item in powerset(small_universe)]
len(all_subsets), [ordered(item) for item in all_subsets]

(8, [[2], [1, 2], [0, 1, 2], [0, 1], [0, 2], [1], [], [0]])

## Step 2 — Verify commutativity

Union and intersection are commutative:

```python
A | B == B | A
A & B == B & A
```

In [119]:
for left in all_subsets:
    for right in all_subsets:
        assert left | right == right | left
        assert left & right == right & left

print("Commutativity verified.")

Commutativity verified.


## Step 3 — Verify a symmetric-difference identity

In [120]:
for left in all_subsets:
    for right in all_subsets:
        assert left ^ right == (
            (left | right) - (left & right)
        )

print("Symmetric-difference identity verified.")

Symmetric-difference identity verified.


## Step 4 — Verify De Morgan's laws

Complements must be taken relative to a declared universe.

In [121]:
for left in all_subsets:
    for right in all_subsets:
        complement_left = small_universe - left
        complement_right = small_universe - right

        assert small_universe - (left | right) == (
            complement_left & complement_right
        )

        assert small_universe - (left & right) == (
            complement_left | complement_right
        )

print("De Morgan's laws verified.")

De Morgan's laws verified.


## Reusable identity checker

In [122]:
def verify_basic_set_identities(
    universe: set[T],
) -> int:
    subsets = [set(item) for item in powerset(universe)]
    checks = 0

    for left in subsets:
        for right in subsets:
            assert left | right == right | left
            assert left & right == right & left
            assert left ^ right == (
                (left | right) - (left & right)
            )

            complement_left = universe - left
            complement_right = universe - right

            assert universe - (left | right) == (
                complement_left & complement_right
            )

            assert universe - (left & right) == (
                complement_left | complement_right
            )

            checks += 5

    return checks

identity_checks = verify_basic_set_identities(
    small_universe
)

identity_checks

320

# Problem 18 — When sets are the wrong tool

Sets are excellent for unique membership.

However, they deliberately discard multiplicity.

Suppose we want to compare the letters in two words.

In [123]:
word_1 = "letter"
word_2 = "teller"

## Step 1 — Compare letter sets

In [124]:
set(word_1), set(word_2), set(word_1) == set(word_2)

({'e', 'l', 'r', 't'}, {'e', 'l', 'r', 't'}, True)

The sets are equal, but this does not prove the words contain the same number of each letter.

Both words use the same distinct letters, but multiplicities may differ.

## Step 2 — Use `Counter` when counts matter

In [125]:
Counter(word_1), Counter(word_2), Counter(word_1) == Counter(word_2)

(Counter({'e': 2, 't': 2, 'l': 1, 'r': 1}),
 Counter({'e': 2, 'l': 2, 't': 1, 'r': 1}),
 False)

## Step 3 — Build an anagram checker

This is included as a best-practice contrast: choose the data structure that matches the question.

In [126]:
def are_anagrams(left: str, right: str) -> bool:
    normalized_left = [
        char.casefold()
        for char in left
        if char.isalnum()
    ]

    normalized_right = [
        char.casefold()
        for char in right
        if char.isalnum()
    ]

    return Counter(normalized_left) == Counter(normalized_right)

assert are_anagrams("Dormitory", "Dirty room")
assert not are_anagrams("letter", "teller")

are_anagrams("The eyes", "They see")

True

# Additional guided mini-examples

## Mini-example 1 — Safe removal

Use `discard` when an element may be absent.

In [127]:
values = {1, 2, 3}

values.discard(4)
values.discard(2)

ordered(values)

[1, 3]

## Mini-example 2 — In-place intersection

Use `intersection_update` when mutation is intentional.

In [128]:
allowed = {"read", "write", "delete", "export"}
plan_limits = {"read", "write"}

allowed.intersection_update(plan_limits)
ordered(allowed)

['read', 'write']

## Mini-example 3 — Method form with a generator

Set methods can consume general iterables.

In [129]:
base = {1, 2, 3, 4, 5}
even_generator = (value for value in range(10) if value % 2 == 0)

ordered(base.intersection(even_generator))

[2, 4]

## Mini-example 4 — Proper subset versus subset

Equality is allowed for `<=`, but not for `<`.

In [130]:
left = {1, 2}
same = {1, 2}
larger = {1, 2, 3}

(left <= same, left < same, left < larger)

(True, False, True)

## Mini-example 5 — Incomparable sets

Set containment is a partial order.

Two sets may be neither subsets nor supersets of one another.

In [131]:
x = {1, 2}
y = {2, 3}

{
    "x_subset_y": x <= y,
    "x_superset_y": x >= y,
    "x_equal_y": x == y,
}

{'x_subset_y': False, 'x_superset_y': False, 'x_equal_y': False}

# Final best-practice checklist

When solving a problem with sets, ask these questions in order:

1. **What is the universe of possible elements?**
2. **What does membership in each input set mean?**
3. **Should duplicates matter?**
4. **Is the desired rule “and,” “or,” “left but not right,” or “exactly one”?**
5. **Do I need exact frequency across more than two sets?**
6. **Should the function mutate an existing set or return a new one?**
7. **Do I need an immutable `frozenset`?**
8. **Am I accidentally depending on display order?**
9. **Have I tested empty inputs?**
10. **Would a `Counter`, list, dictionary, or graph representation be more appropriate?**

In [132]:
# Final notebook-wide regression checks.

assert course_summary["all_three"] == {"Cy", "Eli", "Gus"}
assert members_in_exactly_k_groups(teams, 4) == {"Cy"}
assert rollout["denied_but_active"] == set()
assert partition_report["valid"] is True
assert log_report["at_least_twice"] == {"e2", "e3", "e7", "e9"}
assert recommendations[0][0] == "B"
assert search_result == {1, 5}
assert two_hop["recommendations"] == {"D", "E", "F"}
assert negotiation["missing_required"] == set()
assert config_report["default_only"] == {"port", "timeout"}
assert weighted_graph["C"]["A"] == 9
assert len(minimum_hits) == 2
assert candidate_result == {9}
assert drift_report["persistent"] == {"free", "paid"}
assert identity_checks > 0
assert are_anagrams("The eyes", "They see")

print("All tutorial notebook checks passed.")

All tutorial notebook checks passed.


# Closing summary

The central strength of sets is that they let us express membership rules directly.

A long procedural description such as:

> find all users who belong to either eligible group, remove denied users, and then remove users already active

can become:

```python
eligible_not_active = (group_a | group_b) - denied - active
```

The compactness is valuable only when the membership meaning is understood.

That is why this notebook repeatedly separated each problem into small logical steps before presenting the final reusable solution.